# Get-Zero 机器人资产生成调研

调研 get_zero 中 LeapHand 变体的生成方式，以及对 Allegro/Shadow 的可迁移性。

## 1. 结论摘要

**核心结论**：

- **是脚本生成的**：Get-Zero 通过一个 Python 脚本 `gen_leap_assets.py` 自动生成了 633 个 LeapHand 变体的 URDF + YAML 配置，过程完全参数化、可复现。
- **部分依赖外部建模软件**：
  - **关节拓扑变化（001-236）**：完全由代码控制，不需要任何外部建模。脚本通过组合式枚举各手指的关节链配置（移除/保留不同关节），自动拼装 URDF。复用原始手的 STL mesh。
  - **连杆长度变化（237-633）**：URDF 拼装和关节偏移量由代码控制，但加长版连杆的 STL mesh（如 `mcp_joint_long.stl`）是用 Blender 手工制作的（代码库中存有 `.blend` 源文件）。
- **对 Allegro/Shadow 的可迁移性**：
  - **关节拓扑变化**：理论上可迁移，但需要重写脚本（不同手型的运动链结构、命名规则、关节分布完全不同）。工程量中等。
  - **连杆长度变化**：如果只改 URDF 中的关节偏移，同时仍复用原始 mesh，可以快速做。如果需要物理上精确的加长版 mesh，则需 Blender/CAD 建模。
  - **跨手型泛化训练**：Get-Zero 的方法论高度依赖 LeapHand 的模块化特性（四指结构、关节命名数字化、同构手指）。ShadowHand（24 DOF、underactuated、复杂拇指）和 Allegro（不同运动链拓扑）迁移难度显著更高。

## 2. 资产来源与生成流程

### 2.1 目录结构

| 路径 | 内容 |
|------|------|
| `get_zero/rl/assets/leap/leap_hand/original_refactor.urdf` | 原始 LeapHand URDF（16 DOF 完整版），作为生成的模板 |
| `get_zero/rl/assets/leap/leap_hand/*.stl` | 原始连杆的 STL mesh（`palm_lower.stl`, `mcp_joint.stl`, `pip.stl`, `dip.stl`, `fingertip.stl` 等） |
| `get_zero/rl/assets/leap/leap_hand/custom_models/` | 加长版连杆的 STL + Blender 源文件（`mcp_joint_long.stl`, `dip_long.stl` 等） |
| `get_zero/rl/assets/leap/leap_hand/generated/urdf/` | 生成的 633 个变体 URDF（`001.urdf` ~ `633.urdf`） |
| `get_zero/rl/assets/leap/leap_hand/generated/tree/` | 每个变体的运动树文本可视化 |
| `get_zero/rl/cfg/embodiment/LeapHand/OrigRepo.yaml` | 原始手型的配置参数 |
| `get_zero/rl/cfg/embodiment/LeapHand/generated/` | 633 个变体对应的 YAML 配置（DOF 数、canonical pose、sim/real 索引映射等） |

### 2.2 核心生成脚本

**脚本路径**：`get_zero/rl/scripts/gen_leap_assets.py`

**执行链路**：

```
原始 URDF (original_refactor.urdf)
    ↓ 解析 XML，提取 joint/link 连接关系
    ↓
Phase 1: 关节拓扑变化（001-236）
    ↓ 枚举 4 个手指的关节链组合
    ↓ 每个组合 → 拼装新 URDF + 生成 YAML 配置
    ↓
Phase 2: 连杆长度变化（237-633）
    ↓ 基于 Phase 1 的某些基础变体
    ↓ 枚举哪些连杆使用加长版
    ↓ 替换 mesh 引用 + 修改关节偏移量 → 新 URDF + YAML
    ↓
输出：633 个 URDF + 633 个 YAML + 633 个 tree 文件
```

### 2.3 输入/输出详情

**输入**：
- `original_refactor.urdf`：原始 16-DOF LeapHand 的完整 URDF
- `OrigRepo.yaml`：canonical pose、sim↔real 索引映射
- 原始 STL mesh 文件
- 加长版 STL mesh 文件（手工在 Blender 中制作）

**输出**（每个变体生成三项）：
- URDF 文件：包含该变体的 link/joint 拓扑和 mesh 引用
- YAML 配置：`dofCount`, `canonical_pose`, `sim_to_real_indices`, `fingertip_indices`, `joint_name_to_joint_i`, `embodiment_name`
- Tree 文本：运动链的可视化树形结构

## 3. 参数化可生成的部分 vs 依赖外部建模的部分

### 3.1 完全由代码控制的参数

| 参数 | 控制方式 | 说明 |
|------|----------|------|
| **关节拓扑（父子连接关系）** | 脚本枚举组合 | 每个手指可独立选择保留哪些关节段（如完整链 `mcp→pip→dip→fingertip`，或缩短为 `dip→fingertip`，或只保留 `fingertip`） |
| **关节 origin（位置/旋转）** | 脚本中的 `parent_link_to_child_link_rpy` 字典 | 包含所有可能的 parent-child 连接的 RPY 变换，含原始 URDF 中不存在的新连接 |
| **关节限位、力矩、速度** | 直接继承原 URDF | 从原始 joint XML 中 deepcopy |
| **质量和惯量** | 直接继承原 URDF | 从原始 link XML 中 deepcopy，**未**根据长度变化重新计算 |
| **关节命名** | 保留原始电机 ID 命名 | 关节名即真实电机编号（`"0"` ~ `"15"`），用于 sim↔real 索引映射 |
| **canonical pose** | 脚本中的 `finger_chain_to_canoncial_pose` 字典 | 为新拓扑定义合理的默认手型 |
| **DOF 数、指尖索引** | 脚本自动计算 | 根据生成的拓扑自动统计 |
| **连杆长度变化时的关节偏移** | `long_link_name_to_child_joint_pos` 字典 | 加长连杆后，子关节的 xyz 偏移量需手动测量后写入脚本 |
| **mesh 颜色** | 脚本硬编码 | 加长版连杆统一改为橙色 `rgba="0.878 0.467 0.0 1.0"` |

### 3.2 需要外部建模软件的部分

| 参数 | 外部依赖 | 说明 |
|------|----------|------|
| **原始连杆 mesh（STL）** | 由 SolidWorks/Onshape 导出 | `.part` 文件（SolidWorks 格式）与 `.stl` 文件共存，说明 mesh 来源于 CAD 软件 |
| **加长版连杆 mesh（STL）** | 由 Blender 制作 | `custom_models/` 目录下有 `.blend` 和 `.stl` 文件。脚本注释写到 "measuring the distance from the origin of the STL file (open the STL in Blender...)" |
| **关节偏移量测量** | 在 Blender 中手动测量 | 脚本注释："This value is determined by measuring joint to joint distance in the modified (increase length) link STL file" |

### 3.3 关键判断

**关节拓扑变化**（001-236，约 236 个变体）：
- **100% 由代码自动生成**，不需要任何外部工具
- 复用原始 STL mesh，只改 URDF 的 XML 拼装
- 这是 Get-Zero 论文中 "graph variation" 的核心

**连杆长度变化**（237-633，约 397 个变体）：
- URDF 拼装是自动化的
- 但**加长版 STL mesh 需要 Blender 手工制作**（5 种加长部件：`mcp_joint_long`, `dip_long`, `fingertip_long`, `thumb_dip_long`, `thumb_fingertip_long`）
- 关节偏移量需要在 Blender 中手动测量后硬编码到脚本中
- 不过这只是一次性工作——一旦制作好加长版 mesh 并写好偏移量，后续的组合枚举全部自动化

### 3.4 未参数化的物理属性

值得注意的是，脚本在生成加长版连杆时**没有**重新计算以下物理属性：
- 惯量（inertia）：直接复用原始值
- 质量（mass）：直接复用原始值
- 碰撞体（collision mesh）：引用加长版 STL，但碰撞形状来自该 STL 本身

这意味着仿真中加长版连杆的动力学属性**不完全物理准确**，但对于 RL 训练来说可能影响不大（domain randomization 可以覆盖）。

## 4. Get-Zero 生成方法的技术细节

### 4.1 Phase 1：关节拓扑变化（Graph Variation）

脚本的核心思路是**组合式枚举**：

```python
# 每个手指可以选择不同长度的运动链
index_connectivity_configurations = [
    ['mcp_joint', 'pip', 'dip', 'fingertip'],   # 完整 4 段
    ['mcp_joint', 'pip', 'fingertip'],            # 跳过 dip
    ['dip', 'fingertip'],                          # 只保留 dip 起
    ['fingertip'],                                 # 只有指尖
    []                                             # 整个手指缺失
]

# 拇指只有 2 种配置（完整 4 段或跳过 thumb_dip）
thumb_connectivity_configurations = [
    ['pip_4', 'thumb_pip', 'thumb_dip', 'thumb_fingertip'],
    ['pip_4', 'thumb_pip', 'thumb_fingertip']
]
```

筛选条件：
- 至少保留 2 个手指
- 非拇指手指的关节总数 ≥ 3，或至少有一个非拇指手指有 ≥ 2 个关节

理论组合数：$2 \times 5^3 = 250$，实际经过筛选后生成 236 个变体。

LeapHand 的关键设计特性使这种枚举成为可能：
1. **模块化结构**：三个非拇指手指结构完全同构（只是命名后缀不同 `_2`, `_3`）
2. **数字命名关节**：关节名直接是电机 ID（`"0"` ~ `"15"`），便于 sim↔real 映射
3. **独立的手指电机**：每个关节段对应一个独立电机，没有耦合关节

### 4.2 Phase 2：连杆长度变化（Link Length Extension）

对 Phase 1 中选定的基础变体，枚举哪些连杆使用加长版：

```python
# 可以被加长的连杆
has_long_variant = ['mcp_joint', 'dip', 'fingertip', 'thumb_dip', 'thumb_fingertip']
```

加长方式：
- 替换 URDF 中的 mesh 引用（如 `mcp_joint.stl` → `custom_models/mcp_joint_long.stl`）
- 修改子关节的 origin xyz（加长约 1.5cm）
- 修改 link 的 visual origin（如果需要）

筛选条件：
- 单个手指最多 2 个连杆被加长
- 控制总修改数量的上下限
- 要求至少有一定数量的非指尖修改
- 洗牌后只取 1 个变体（对于非 001 基础手型）

### 4.3 运动链示例对比

**001（完整 16 DOF，与原始相同）**：
```
palm_lower
├── index:  1→mcp_joint→0→pip→2→dip→3→fingertip
├── thumb:  12→pip_4→13→thumb_pip→14→thumb_dip→15→thumb_fingertip
├── middle: 5→mcp_joint_2→4→pip_2→6→dip_2→7→fingertip_2
└── ring:   9→mcp_joint_3→8→pip_3→10→dip_3→11→fingertip_3
```

**100（缺失食指，环指只保留 dip+fingertip，10 DOF）**：
```
palm_lower
├── thumb:  12→pip_4→13→thumb_pip→14→thumb_dip→15→thumb_fingertip
├── middle: 5→mcp_joint_2→4→pip_2→6→dip_2→7→fingertip_2
└── ring:   9→dip_3→11→fingertip_3
```

## 5. 对 Allegro / ShadowHand 的可迁移性分析

### 5.1 TRO-Grasp 中已有的多手型变体

工作区中的 `TRO-Grasp` 项目已经为 Allegro 和 ShadowHand 做了一些 URDF 变体，可以作为参考：

- **Allegro**：`allegro_hand_left_extended_joints_{5-9}.urdf`（减少关节数）、`allegro_hand_left_extended_scaled_{11-15}.urdf`（整体缩放）
- **ShadowHand**：同样有 `joints_{5-9}` 和 `scaled_{11-15}` 变体

但这些变体似乎是**手工创建的 URDF 文件**，不像 Get-Zero 有一套系统化的生成脚本。

### 5.2 迁移到 Allegro Hand 的分析

**Allegro Hand 特性**（相比 LeapHand）：
- 16 DOF，4 指（同 LeapHand）
- 但运动链结构不同：每个手指 4 个关节（与 LeapHand 类似但命名和拓扑不同）
- 拇指结构差异较大
- mesh 格式不同

**可直接复用的**：
- Phase 1 式的关节拓扑枚举**设计思路**可复用
- EmbodimentProperties 类（邻接矩阵、图结构构建）对 URDF 格式通用

**需要重写的**：
- 生成脚本需要针对 Allegro 的连接关系重写（不同的 link/joint 命名规则）
- canonical pose 需要重新定义
- sim↔real 索引映射不适用（Allegro 的关节命名不是数字）
- 加长版 mesh 需要基于 Allegro 的 mesh 重新制作

**工程量估计**：中等。核心逻辑（组合枚举 + URDF 拼装）可复用，但需要：
1. 分析 Allegro URDF 的运动链结构
2. 定义 Allegro 的关节链配置空间
3. 写 Allegro 专用的关节变换字典
4. 定义 canonical pose

### 5.3 迁移到 ShadowHand 的分析

**ShadowHand 特性**：
- **24 DOF**（远多于 LeapHand 的 16 DOF）
- **5 个手指**（含小指）
- **欠驱动关节**（underactuated）：部分手指关节是耦合的
- 拇指结构非常复杂
- 有前臂部件

**困难点**：
- 关节耦合（coupled joints）使得"移除单个关节"变得更复杂
- 运动链更深更复杂
- 欠驱动关系需要在配置中额外处理
- DOF 数量差异大，训练时的 observation/action 空间差异显著

**工程量估计**：较高。除了 Allegro 所需的所有工作外，还需要：
1. 处理耦合关节的逻辑
2. 更大的拓扑枚举空间
3. 可能需要重新考虑变体生成的约束条件

### 5.4 跨手型泛化训练的可行性

从 **cross-embodiment 训练**角度：

| 维度 | LeapHand 变体间 | LeapHand → Allegro | LeapHand → ShadowHand |
|------|-----------------|--------------------|-----------------------|
| 拓扑结构 | 同族变体，子图关系 | 不同但结构相似 | 差异大 |
| DOF 数 | 变化范围 4-16 | 需要适配 | 需要跨越 16→24 |
| 驱动方式 | 全驱动 | 全驱动 | 欠驱动 |
| mesh 几何 | 共享同一套 mesh | 完全不同 | 完全不同 |
| 运动学参数 | 同一参数空间 | 不同运动学参数 | 不同运动学参数 |

Get-Zero 的 Graph Embodiment Transformer (GET) 架构在设计上**可以**处理不同图结构，因为它使用**图注意力机制**（基于运动链邻接关系的注意力 bias）。但训练数据目前**仅限 LeapHand 变体**，跨手型泛化需要：
1. 为每种手型训练 expert policy
2. 收集 demonstration data
3. 用新的 embodiment tokenization 重新训练 GET

## 6. 实践建议

### 6.1 最现实的技术路线

**Phase A（最省工程量 — 同拓扑参数化变体）**：

1. 只改**关节限位、连杆长度（URDF 中的 joint origin offset）、质量/惯量**
2. **不改 mesh**——视觉效果不完全匹配物理参数，但仿真动力学有效
3. 对 LeapHand / Allegro / ShadowHand 各自生成一批同拓扑但参数不同的变体
4. 直接导入 IsaacLab 训练

**优点**：工程量最小，不需要任何外部建模工具，纯 Python XML 操作
**缺点**：视觉-物理不一致；不涉及拓扑变化

**Phase B（中等工程量 — 关节拓扑变体）**：

1. 参考 Get-Zero 的 Phase 1 方法，为每种手型编写拓扑枚举脚本
2. 核心工作：分析目标手型的运动链 → 定义合法的子图配置 → 编写拼装逻辑
3. 复用原始 mesh，只改 URDF 结构
4. 为每个变体定义 canonical pose 和关节映射

**优点**：产生真正不同的 embodiment 图结构，与 Get-Zero 方法论直接对齐
**缺点**：每种手型需要独立编写生成脚本（约 1-2 天/手型）

**Phase C（高工程量 — 加上 mesh 变化）**：

1. 在 Blender 中为每种手型制作加长版连杆 mesh
2. 测量关节偏移量
3. 写入生成脚本

**优点**：最完整的变体空间
**缺点**：每种手型需要 Blender 建模工作，可能需要 3D 建模经验

### 6.2 建议优先级

```
[推荐] Phase A → Phase B → Phase C
       ─────→  ───────→  ────────→
       1天        1-2天/手型   依赖 Blender
```

对于**科研验证**阶段，建议：
- 先做 **Phase A**（纯参数化，同拓扑变体），快速验证 cross-embodiment 训练流程是否能工作
- 如果论文需要展示**拓扑泛化**能力，再做 **Phase B**
- Phase C 的优先级最低，除非论文需要展示连杆长度变化的真实 mesh

### 6.3 最容易卡住的环节

| 环节 | 风险 | 建议 |
|------|------|------|
| 为新手型写拓扑枚举脚本 | 中等——需要仔细理解 URDF 结构 | 直接参考 `gen_leap_assets.py` 的代码结构 |
| 加长版 mesh 制作 | 高——需要 3D 建模技能 | 可以跳过，用简单几何体（圆柱/盒体）替代原始 mesh |
| canonical pose 定义 | 低——可以用零位或物理合理的默认值 | 可通过仿真快速迭代 |
| ShadowHand 的耦合关节处理 | 高——逻辑复杂 | 考虑先标记耦合关节，确保移除时保持物理一致性 |
| IsaacLab 集成 | 中等——需要适配 ManagerBasedRLEnv 的资产加载方式 | 利用 AnyMani 已有的 LeapHand 加载逻辑作为模板 |

## 7. 澄清：Get-Zero 并没有为每个变体手动创建 mesh

**用户反馈**：担心每个变体都需要手动创建 mesh

**分析**：

这是一个很重要的澄清点。**Get-Zero 并没有为 633 个变体分别创建 mesh**。实际情况远比这简单：

### 7.1 mesh 复用机制

整个 Get-Zero 的 633 个变体，总共只用到了以下 **13 个** STL mesh 文件：

**原始 mesh（8 个，来自 LeapHand 原始 CAD）**：
1. `palm_lower.stl` — 手掌
2. `mcp_joint.stl` — MCP 关节段
3. `pip.stl` — PIP 关节段
4. `dip.stl` — DIP 关节段
5. `fingertip.stl` — 指尖
6. `thumb_pip.stl` — 拇指 PIP
7. `thumb_dip.stl` — 拇指 DIP
8. `thumb_fingertip.stl` — 拇指指尖

**加长版 mesh（5 个，在 Blender 中制作）**：
9. `mcp_joint_long.stl`
10. `dip_long.stl`
11. `fingertip_long.stl`
12. `thumb_dip_long.stl`
13. `thumb_fingertip_long.stl`

### 7.2 为什么不需要更多 mesh？

**Phase 1（关节拓扑变化，001-236）**：
- 这些变体**完全不需要新 mesh**
- 它们只是从原始手型中**移除某些关节和连杆**
- 被保留的连杆直接引用原始 STL 文件
- 例如变体 100（缺食指，环指只有 dip+fingertip）：
  - 保留下来的 `dip_3` 依然引用 `dip.stl`
  - 保留下来的 `thumb_fingertip` 依然引用 `thumb_fingertip.stl`

**Phase 2（连杆长度变化，237-633）**：
- 只需要 5 个加长版 mesh（一次性在 Blender 中制作）
- 所有使用加长版的变体都**引用同一个** `xxx_long.stl` 文件
- 例如变体 240 可能是 `mcp_joint_long` + 其他正常长度的组合
- 变体 250 可能是 `dip_long` + `fingertip_long` 的组合
- 但它们引用的都是同一批 STL

### 7.3 所以总工程量是多少？

| 工作项 | 数量 | 方式 |
|--------|------|------|
| 原始 mesh | 8 个 | 已有（LeapHand 原始 CAD 导出） |
| 加长 mesh | 5 个 | Blender 中拉伸原始 mesh（一次性工作） |
| 拓扑变体 URDF | 236 个 | 脚本自动生成 |
| 长度变体 URDF | 397 个 | 脚本自动生成 |
| 配置 YAML | 633 个 | 脚本自动生成 |

**所以只有 5 个加长版 mesh 需要手工做，其余全部自动化。**

### 7.4 如果迁移到其他手型

如果要给 Allegro Hand 做类似的变体：
- **Phase A（纯参数化）**：**0 个新 mesh**——只改 URDF 中的数值参数
- **Phase B（关节拓扑变化）**：**0 个新 mesh**——移除关节/连杆时复用原始 mesh
- **Phase C（连杆长度变化）**：需要为 Allegro 的几种连杆类型各做 **1 个**加长版 mesh

对于 Allegro（4 指，每指 3-4 种连杆类型），大约需要制作 **4-6 个**加长版 STL。

## 8. mesh 拉伸能否用代码实现？

**用户反馈**：能不能不用 Blender，纯代码拉伸 mesh？

**分析**：

### 8.1 短回答：可以，而且并不难

Python 中有成熟的 3D mesh 操作库，完全可以用代码实现 mesh 拉伸。Get-Zero 作者选择 Blender 可能是因为需要精确控制拉伸效果和验证视觉结果，但从技术上讲，代码方案完全可行。

### 8.2 代码拉伸方案

**方法 1：顶点坐标缩放（最简单）**

```python
import trimesh

mesh = trimesh.load('dip.stl')
# 沿 Y 轴（连杆长轴方向）拉伸 1.5cm
# 找到 Y 坐标 > 某个阈值的顶点，增加偏移
stretch_axis = 1  # 假设 Y 轴是连杆方向
threshold = mesh.vertices[:, stretch_axis].mean()
mask = mesh.vertices[:, stretch_axis] > threshold
mesh.vertices[mask, stretch_axis] += 0.015  # 1.5cm
mesh.export('dip_long.stl')
```

**方法 2：仿射变换（沿某轴等比缩放）**

```python
import trimesh
import numpy as np

mesh = trimesh.load('dip.stl')
# 沿连杆方向缩放 1.3 倍
scale_matrix = np.eye(4)
scale_matrix[1, 1] = 1.3  # Y 轴方向缩放
mesh.apply_transform(scale_matrix)
mesh.export('dip_long.stl')
```

**方法 3：OBB/局部变形（更精确）**

```python
import trimesh

mesh = trimesh.load('dip.stl')
# 用 OBB 确定主轴方向，沿主轴拉伸
obb = mesh.bounding_box_oriented
# 沿 OBB 的最长轴方向缩放
# ...更精确的变形控制
```

### 8.3 Get-Zero 作者为什么用 Blender？

从代码库的蛛丝马迹推测：

1. **精确控制关节中心位置**：拉伸后需要确保关节连接处的几何体精确对齐。Blender 的可视化交互更直观
2. **制作 3D 打印部件**：`custom_models/3d_printing/` 目录下有打印用的 STL，说明这些加长部件需要**实物制作**。实物部件对几何精度要求更高，用 Blender 手工调整更可靠
3. **一次性工作量不大**：只有 5 个 mesh 需要做，不值得写自动化脚本

### 8.4 对我们的场景的建议

如果我们的目标是**仿真中的 RL 训练**（不需要实物制作），用代码拉伸 mesh 完全可行：

| 方案 | 适用场景 | 精度 |
|------|----------|------|
| trimesh 顶点偏移 | 简单连杆拉伸 | 中等 |
| 仿射缩放 | 整体等比缩放 | 高 |
| 不改 mesh，只改 URDF scale | 最简单但物理属性需配合调整 | 低 |
| 用简单几何体（圆柱/盒体）替代原始 mesh | 追求最大简化 | 仿真足够 |

**推荐**：如果只是做科研验证，最极端的简化方案是**完全不改 mesh**——只调整 URDF 中的关节偏移量来改变运动学行为。仿真中连杆的碰撞体和视觉 mesh 不精确匹配关节偏移，对 RL 训练影响不大（尤其在有 domain randomization 的情况下）。

## 9. URDF vs MJCF vs USD：哪种格式最适合生成变体？

**用户反馈**：URDF 有 Get-Zero 参考；MJCF 结构更递归简约；USD 是二进制不好提取特征。哪个最合适？另外代码拉伸后关节连接处的几何体精确对齐是否可行？

### 9.1 三种格式对比

| 维度 | URDF | MJCF | USD |
|------|------|------|-----|
| **可读性** | XML，结构清晰 | XML，更紧凑递归 | 二进制 / ASCII，复杂 |
| **代码操作** | `xml.etree` / `yourdfpy` | `xml.etree` / `dm_control` | `pxr.Usd` API，学习成本高 |
| **特征提取** | 容易：joint limits, axis, origin 直接解析 | 容易：同为 XML | 困难：需通过 USD API 遍历 prim |
| **Get-Zero 参考** | 完整示例 | 无 | 无 |
| **Isaac Lab 支持** | 原生支持（通过 USD 转换） | 原生支持（通过 USD 转换） | 原生格式 |
| **生态 / 社区** | 最广泛 | MuJoCo 生态 | NVIDIA 生态 |
| **运动链修改** | 需手动管理 parent/child | 自动继承，body 嵌套即表达树型 | 需操作 USD prim 层级 |

### 9.2 建议：用 URDF 作为生成格式

理由：

1. **有成熟参考**：Get-Zero 的 `gen_leap_assets.py` 提供了完整的 URDF 生成模板，你在 plan.ipynb 中的 static token 提取逻辑（joint limits, axis, rest-pose transform, link geometry）直接从 URDF 解析即可。

2. **特征提取友好**：plan.ipynb 中定义的 $x_j^{stat}$ 需要从描述文件中提取 $q_{\min,j}, q_{\max,j}, a_j, T_{p\to j}^{rest}, g(l_j), d_j^{topo}$ 等。URDF 的 XML 结构对这些字段的提取最直接。

3. **Isaac Lab 的 URDF→USD 转换是自动化的**：Isaac Lab 内部通过 `omni.isaac.lab.sim.converters.UrdfConverter` 自动将 URDF 转为 USD 用于仿真。你不需要手动维护 USD 文件。

4. **MJCF 的递归结构虽然优雅，但不利于组合枚举**：Get-Zero 的方法核心是"枚举所有合法的关节子集 → 拼装新 URDF"。URDF 的扁平结构（所有 joint/link 是 robot 元素的直接子节点）更适合这种拼装式生成。MJCF 的嵌套 body 结构在"任意移除中间节点"时需要递归调整嵌套关系，更容易出错。

**工作流建议**：
```
URDF（变体生成 + 特征提取）→ Isaac Lab 自动转 USD → 仿真训练
```

### 9.3 关节连接处的几何对齐：代码能否做到？

**短回答**：对于仿真 RL 训练，**完全不需要精确对齐**。

**原因分析**：

需要区分两个层面：

**1. 运动学层面的对齐（关节偏移量）**：
- 这完全由 URDF 中的 `<joint><origin xyz="..." rpy="..."/>` 控制
- 与 mesh 几何体无关
- 代码可以精确控制，没有任何困难

**2. 视觉/碰撞几何层面的对齐（mesh 边界匹配）**：
- 这是"拉伸 mesh 后，两段连杆的 STL 边界是否无缝拼接"的问题
- 对 RL 训练来说，**碰撞体才重要**，而碰撞体可以用简单凸包或盒体近似
- 视觉 mesh 不影响物理仿真

**实际影响**：

| 层面 | 不对齐的后果 | 是否影响 RL |
|------|-------------|------------|
| 运动学 | 关节位置错误 → 手的形态错误 | **严重影响** |
| 碰撞体 | 可能有穿透或间隙 | 轻微影响（domain randomization 可覆盖） |
| 视觉 mesh | 渲染时看起来有缝隙 | **不影响** |

因此，最务实的做法是：

1. **运动学对齐**：代码精确控制 joint origin → 100% 准确
2. **碰撞体**：可以用简单几何体（盒体/球体/胶囊体）替代原始 mesh → 不需要 Blender
3. **视觉 mesh**：不改原始 mesh，接受视觉上的不匹配 → 不影响训练

或者更进一步：如果用 URDF 的 `<mesh scale="sx sy sz"/>` 属性，可以在不修改 STL 文件的情况下缩放 mesh。虽然这只能做等比或轴向缩放（不能做局部拉伸），但对"让连杆看起来更长"已经足够。

### 9.4 对 plan.ipynb 中方法的影响

从 plan.ipynb 的方法设计来看，static token 提取需要：
- $g(l_j)$：link 几何编码 — 可以从 URDF 中读取 mesh 并用 BPS 或 bbox 编码
- $T_{p\to j}^{rest}$：rest-pose 变换 — 直接从 URDF joint origin 获取
- $\Delta T_{ij}^{rest}$：关节间相对 SE(3) — 通过 FK 计算

这些**全部可以从 URDF 中纯代码提取**，无需依赖 mesh 是否对齐。`$g(l_j)$` 如果用 bbox/尺度编码而非 BPS，甚至不需要加载 mesh 文件。

## 10. 技术验证：代码精确控制 URDF 缩放

**用户反馈**：mesh 拉伸能否用代码做，不依赖 Blender？能否保证 visual/collision 对齐？请做一个 2R 机器人示例。

**验证结果**：已在 `example/` 目录下生成完整示例，代码能精确控制。

### 10.1 生成的文件

| 文件 | 说明 |
|------|------|
| `example/original_2r.urdf` | 原始 2R 机器人（link1=0.3m, link2=0.2m） |
| `example/scaled_2r.urdf` | 缩放后（link1 沿 Z 轴 ×1.5 → 0.45m） |
| `example/gen_scaled_urdf.py` | 生成脚本 |

### 10.2 验证结果

```
原始 URDF 对齐验证：
  base_link: ✓ 对齐
  link1:     ✓ 对齐
  link2:     ✓ 对齐

缩放后 URDF 对齐验证：
  base_link: ✓ 对齐
  link1:     ✓ 对齐
  link2:     ✓ 对齐
```

参数自动更新：
- link1 长度: 0.3m → 0.45m（×1.5）
- link1 质心: z=0.15 → z=0.225（随长度等比）
- link1 质量: 1.018 kg → 1.527 kg（密度不变假设）
- link1 惯量: 基于新的长度和质量重新计算
- joint2 位置: z=0.3 → z=0.45（link1 末端）

### 10.3 对齐保证的数学原理

代码能保证对齐的原因非常直接：

**对于 URDF primitive（cylinder/box/sphere）**：
- visual 和 collision 的 geometry 参数和 origin 使用**完全相同的缩放逻辑**
- 只要原始 URDF 中两者的参数一致，缩放后必然一致
- 这是纯数值运算，没有任何近似误差

**对于外部 STL mesh**：
- 方案 A：在 URDF 中用 `<mesh scale="sx sy sz"/>`，visual 和 collision 的 mesh 引用同一个文件、使用相同 scale → 必然对齐
- 方案 B：用 trimesh 修改 STL 顶点，对 visual 和 collision 的 STL 执行相同操作 → 必然对齐
- 关键：只要对 visual 和 collision **施加相同的变换**，对齐就有保证

### 10.4 脚本的核心函数

`scale_link_along_axis()` 自动处理以下所有更新：

1. **geometry 缩放**：cylinder length / box size / mesh scale
2. **origin 更新**：visual、collision、inertial 的质心位置
3. **物理参数重算**：质量（密度 × 体积）、惯量（圆柱/盒体公式）
4. **子关节偏移**：以该 link 为 parent 的所有子 joint 的 origin xyz

### 10.5 结论

**代码精确控制 URDF 缩放是完全可行的，且能保证 visual/collision 对齐。** 这意味着：

- 不需要 Blender 来做连杆长度变体
- 整个变体生成流程可以端到端自动化
- 适合批量生成和实验迭代
- 适合未来扩展到 LeapHand / Allegro / ShadowHand